In [61]:
import pandas as pd
import re
import string
import numpy as np

tweets = [
"the new pokemon anime reboot looks so fresh loving the new art style",
"why is catching a shiny pokemon so hard",
"just started playing scarlet again forgot how beautiful paldea is",
"ash leaving the series still doesnt feel real",
"pikachu will always be the face of pokemon no debate",
"the competitive scene feels unbalanced this year",
"can we talk about how good the legends arceus story was",
"this new pokemon event raid is too difficult omg",
"i miss the old games sometimes simpler times",
"caught my first shiny today im screaming",
"the mobile pokemon games are fun but too many microtransactions",
"charizard literally gets too much attention",
"when is game freak going to fix performance issues",
"i love exploring open worlds in pokemon so relaxing",
"pokecommunity is hyped today lol",
"greninja deserved better movesets this gen",
"pokemon sleep is actually helping me fix my bedtime",
"gardevoirs design is honestly so beautiful",
"the new tcg cards look insane omg",
"pokemon unite matchmaking makes no sense",
"why does every rival have a broken team except me",
"the soundtrack of every pokemon game hits so hard",
"some new pokemon designs are getting weird",
"eevee evolutions are the best part of the franchise",
"pokemon go still has my heart after all these years",
"i want a remake of black and white so bad",
"nintendo please drop a new pokemon trailer already",
"the new legendary pokemon is kinda mid",
"pokemon fanbase arguing again nothing new",
"i love cozy nights playing pokemon on my switch",
"my starter fainted five times today im done",
"flying through the map glitch was hilarious",
"trading pokemon with randoms is surprisingly fun",
"team rocket will always be iconic",
"my whole childhood was pokemon and im proud of it",
"some gyms are way harder than they should be",
"why is leveling so slow in this game omg",
"the anime soundtrack brings back nostalgia",
"pokemon challenges make the game way more fun",
"this fan art of lucario i found is amazing",
"my shiny luck is actually insane today",
"the new pokemon animations look stiff",
"wish they would bring mega evolution back",
"pokemon lore goes deeper than people think",
"i got destroyed in ranked battles lol",
"the new dlc wasnt worth the price tbh",
"pokemon soundtracks never disappoint",
"still waiting for an open world pokemon mmo",
"the new paradox pokemon design is fire",
"pokemon cafe remix is cuter than i expected",
"i hate catching pokemon with low catch rates",
"just finished the pokedex feeling proud",
"battle tower is stressing me out again",
"pokemon go community days are always fun",
"this franchise is literally never dying",
"who else resets 20 times just to get a good nature",
"the anime art downgrade was disappointing",
"finally beat the final boss what a fight",
"pokemon snap deserved more updates",
"my switch is overheating because of pokemon again",
"pokemon merch is getting too expensive",
"my friend traded me a shiny umbreon best day ever",
"i cant believe some people skip dialogues",
"pokemon teams take forever to plan",
"this gym puzzle was actually annoying",
"why is every water type pokemon so cool",
"i miss the fairy gym leaders",
"catching legendary pokemon feels amazing",
"some npc trainers are overpowered for no reason",
"pokemon home takes too long to update",
"i cant decide which starter to pick ahhh",
"raid battles are more fun with friends",
"pokemon breeding takes way too long",
"got my dream team ready",
"nuzlockes are stressful but fun",
"the new region looks inspired by europe again",
"pokemon worldbuilding is seriously underrated",
"i failed a shiny because i misclicked",
"pokemon battles feel too easy now",
"this music reminds me of childhood memories",
"why does pikachu get special treatment every gen",
"the anime reboot protagonist is growing on me",
"some people take pokemon battles way too seriously",
"pokemon scarlet glitches were funny",
"my storage box is full again send help",
"this pokemon gym battle theme goes hard",
"i found a rare pokemon by accident lol",
"pokemon go servers down again classic",
"game freak needs more time for development",
"playing pokemon before bed hits different",
"this trainer battle made me rage quit",
"pokemon lore videos are too addicting",
"caught a shiny today and im still shaking",
"the map design is kinda empty this gen",
"wish we had more dragon types",
"pokemon community sometimes too toxic",
"finally evolved my starter took forever",
"pokemon battles feel so nostalgic",
"new pokemon leaks are everywhere",
"i will never stop loving this franchise"
]


df = pd.DataFrame({"Tweet": tweets})

sentiments = ["positive", "negative", "neutral"]
df["Sentiment"] = np.random.choice(sentiments, size=len(df))
df.to_csv("tweets_raw.csv", index=False)

print("Raw dataset created: tweets_raw.csv")
df.head()

Raw dataset created: tweets_raw.csv


,Tweet,Sentiment
0,the new pokemon anime reboot looks so fresh lo...,neutral
1,why is catching a shiny pokemon so hard,positive
2,just started playing scarlet again forgot how ...,positive
3,ash leaving the series still doesnt feel real,neutral
4,pikachu will always be the face of pokemon no ...,positive


In [62]:
df = pd.read_csv("tweets_raw.csv")

print("Loaded labeled dataset:")
df.head()

Loaded labeled dataset:


,Tweet,Sentiment
0,the new pokemon anime reboot looks so fresh lo...,neutral
1,why is catching a shiny pokemon so hard,positive
2,just started playing scarlet again forgot how ...,positive
3,ash leaving the series still doesnt feel real,neutral
4,pikachu will always be the face of pokemon no ...,positive


In [63]:
def clean(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

df["CleanTweet"] = df["Tweet"].apply(clean)

In [64]:
df["label"] = df["Sentiment"].map({
    "positive": 2,
    "neutral": 1,
    "negative": 0
})

In [65]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["CleanTweet"], df["label"], test_size=0.2, random_state=42
)

In [66]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [67]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

y_pred_nb = nb.predict(X_test_tfidf)
print("\n===== NAIVE BAYES =====\n")
print(classification_report(y_test, y_pred_nb))


===== NAIVE BAYES =====

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.43      0.50      0.46         6
           2       0.50      0.71      0.59         7

    accuracy                           0.40        20
   macro avg       0.31      0.40      0.35        20
weighted avg       0.30      0.40      0.34        20



In [68]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)
print("\n===== LOGISTIC REGRESSION =====\n")
print(classification_report(y_test, y_pred_lr))


===== LOGISTIC REGRESSION =====

              precision    recall  f1-score   support

           0       0.17      0.14      0.15         7
           1       0.40      0.33      0.36         6
           2       0.44      0.57      0.50         7

    accuracy                           0.35        20
   macro avg       0.34      0.35      0.34        20
weighted avg       0.33      0.35      0.34        20



In [69]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_tfidf, y_train)

y_pred_svm = svm.predict(X_test_tfidf)
print("\n===== SVM =====\n")
print(classification_report(y_test, y_pred_svm))


===== SVM =====

              precision    recall  f1-score   support

           0       0.17      0.14      0.15         7
           1       0.50      0.50      0.50         6
           2       0.38      0.43      0.40         7

    accuracy                           0.35        20
   macro avg       0.35      0.36      0.35        20
weighted avg       0.34      0.35      0.34        20



In [70]:
df.to_csv("final_processed_data.csv", index=False)
print("Saved final_processed_data.csv")

Saved final_processed_data.csv
